In [1]:
from __future__ import annotations

import os
import json
from getpass import getpass

import requests
import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import MetadataQuery, Filter

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
def json_print(data):
    print(json.dumps(data, indent=2, default=str))

In [5]:
def load_tiny_jeopardy():
    url = "https://raw.githubusercontent.com/weaviate-tutorials/quickstart/main/data/jeopardy_tiny.json"
    return requests.get(url, timeout=30).json()


def load_jeopardy_1k():
    url = "https://raw.githubusercontent.com/weaviate-tutorials/intro-workshop/main/data/jeopardy_1k.json"
    return requests.get(url, timeout=30).json()

In [6]:
data = load_tiny_jeopardy()

print(type(data), len(data))
json_print(data[0])

<class 'list'> 10
{
  "Category": "SCIENCE",
  "Question": "This organ removes excess glucose from the blood & stores it as glycogen",
  "Answer": "Liver"
}


In [7]:
def recreate_question_vector_collection(client, *, include_round=False):
    collection_name = "QuestionVector"

    if client.collections.exists(collection_name):
        client.collections.delete(collection_name)

    properties = [
        wvc.config.Property(
            name="question",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="answer",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="category",
            data_type=wvc.config.DataType.TEXT,
        ),
    ]

    if include_round:
        properties.append(
            wvc.config.Property(
                name="round",
                data_type=wvc.config.DataType.TEXT,
            )
        )

    return client.collections.create(
        name=collection_name,
        vector_config=wvc.config.Configure.Vectors.text2vec_openai(
            model="text-embedding-3-small"
        ),
        properties=properties,
    )

In [8]:
questions = recreate_question_vector_collection(client)

print(questions.name)

QuestionVector


In [9]:
def import_questions(collection, data, *, include_round=False):
    objects = []

    for item in data:
        obj = {
            "question": item["Question"],
            "answer": item["Answer"],
            "category": item["Category"],
        }

        if include_round:
            obj["round"] = item.get("Round", "")

        objects.append(obj)

    result = collection.data.insert_many(objects)

    if result.errors:
        print("Import errors:")
        json_print(result.errors)
    else:
        print("Import completed.")

    return result

In [10]:
import_questions(questions, data)

count = questions.aggregate.over_all(total_count=True)

print("Objects:", count.total_count)

Import completed.
Objects: 10


In [11]:
response = questions.query.fetch_objects(
    limit=3,
    return_properties=["question", "answer", "category"],
)

for obj in response.objects:
    json_print(obj.properties)

{
  "answer": "Elephant",
  "category": "ANIMALS",
  "question": "It's the only living mammal in the order Proboseidea"
}
{
  "answer": "the atmosphere",
  "category": "SCIENCE",
  "question": "Changes in the tropospheric layer of this are what gives us weather"
}
{
  "answer": "Liver",
  "category": "SCIENCE",
  "question": "This organ removes excess glucose from the blood & stores it as glycogen"
}


In [12]:
response = questions.query.fetch_objects(
    limit=1,
    return_properties=["question", "answer", "category"],
    include_vector=True,
)

for obj in response.objects:
    json_print(obj.properties)
    print("Vector length:", len(obj.vector["default"]))
    print("First 10 vector values:", obj.vector["default"][:10])

{
  "answer": "Elephant",
  "category": "ANIMALS",
  "question": "It's the only living mammal in the order Proboseidea"
}
Vector length: 1536
First 10 vector values: [0.07806396484375, 0.00421142578125, 0.00981903076171875, 0.042724609375, -0.060638427734375, 0.063720703125, 0.00020241737365722656, 0.00714874267578125, -0.00574493408203125, 0.00897979736328125]


## Vector search / semantic search: biology

In [13]:
response = questions.query.near_text(
    query="biology",
    limit=2,
    return_properties=["question", "answer", "category"],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    json_print(obj.properties)
    print("Distance:", obj.metadata.distance)
    print("-" * 80)

{
  "category": "SCIENCE",
  "answer": "DNA",
  "question": "In 1953 Watson & Crick built a model of the molecular structure of this, the gene-carrying substance"
}
Distance: 0.7521905899047852
--------------------------------------------------------------------------------
{
  "answer": "species",
  "category": "SCIENCE",
  "question": "2000 news: the Gunnison sage grouse isn't just another northern sage grouse, but a new one of this classification"
}
Distance: 0.7812235951423645
--------------------------------------------------------------------------------


In [14]:
response = questions.query.near_text(
    query="animals",
    limit=10,
    return_properties=["question", "answer", "category"],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print(
        obj.metadata.distance,
        obj.properties["question"],
        "=>",
        obj.properties["answer"],
    )

0.5799150466918945 It's the only living mammal in the order Proboseidea => Elephant
0.6252744197845459 Weighing around a ton, the eland is the largest species of this animal in Africa => Antelope
0.660226047039032 The gavial looks very much like a crocodile except for this bodily feature => the nose or snout
0.7013884782791138 Heaviest of all poisonous snakes is this North American rattlesnake => the diamondback rattler
0.7167699337005615 2000 news: the Gunnison sage grouse isn't just another northern sage grouse, but a new one of this classification => species
0.7987968325614929 This organ removes excess glucose from the blood & stores it as glycogen => Liver
0.824963390827179 Changes in the tropospheric layer of this are what gives us weather => the atmosphere
0.8280520439147949 In 1953 Watson & Crick built a model of the molecular structure of this, the gene-carrying substance => DNA
0.828243613243103 A metal that is ductile can be pulled into this while cold & under pressure => wir

In [15]:
response = questions.query.near_text(
    query="animals",
    distance=0.24,
    limit=10,
    return_properties=["question", "answer", "category"],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print(
        obj.metadata.distance,
        obj.properties["question"],
        "=>",
        obj.properties["answer"],
    )

In [16]:
data_1k = load_jeopardy_1k()

print(len(data_1k))
json_print(data_1k[0])

1000
{
  "Air Date": "2006-11-08",
  "Round": "Double Jeopardy!",
  "Value": 800,
  "Category": "AMERICAN HISTORY",
  "Question": "Abraham Lincoln died across the street from this theatre on April 15, 1865",
  "Answer": "Ford's Theatre (the Ford Theatre accepted)"
}


In [17]:
questions = recreate_question_vector_collection(
    client,
    include_round=True,
)

print(questions.name)

QuestionVector


In [18]:
import_questions(
    questions,
    data_1k,
    include_round=True,
)

count = questions.aggregate.over_all(total_count=True)

print("Objects:", count.total_count)

Import completed.
Objects: 1000


In [19]:
response = questions.query.near_text(
    query="spicy food",
    limit=4,
    return_properties=["question", "answer", "round"],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    json_print(obj.properties)
    print("-" * 80)

Distance: 0.5982016921043396
{
  "answer": "tripe",
  "question": "Popular in Pennsylvania, pepper pot is a peppery soup made from this stomach lining",
  "round": "Jeopardy!"
}
--------------------------------------------------------------------------------
Distance: 0.6257262229919434
{
  "answer": "Chiles Rellenos",
  "question": "The name of this Mexican dish made with chiles & cheese translates to \"stuffed peppers\"",
  "round": "Jeopardy!"
}
--------------------------------------------------------------------------------
Distance: 0.6451320052146912
{
  "answer": "Yellow",
  "question": "A wax bean is a variety of green bean that's this color",
  "round": "Double Jeopardy!"
}
--------------------------------------------------------------------------------
Distance: 0.6586971282958984
{
  "answer": "licorice",
  "question": "Herbs anise & fennel resemble the flavor of this common black candy",
  "round": "Jeopardy!"
}
--------------------------------------------------------------

In [20]:
response = questions.query.near_text(
    query="spicy food recipes",
    filters=Filter.by_property("round").equal("Double Jeopardy!"),
    limit=4,
    return_properties=["question", "answer", "round"],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    json_print(obj.properties)
    print("-" * 80)

Distance: 0.6970611810684204
{
  "answer": "Yellow",
  "question": "A wax bean is a variety of green bean that's this color",
  "round": "Double Jeopardy!"
}
--------------------------------------------------------------------------------
Distance: 0.7018082141876221
{
  "answer": "\"Jambalaya\"",
  "question": "This Creole concoction of meat & seafood is so good Hank Williams wrote a song about it in 1952",
  "round": "Double Jeopardy!"
}
--------------------------------------------------------------------------------
Distance: 0.7040959596633911
{
  "answer": "English/burpless/seedless",
  "question": "This type of cucumber is only grown under artificial conditions in hothouses",
  "round": "Double Jeopardy!"
}
--------------------------------------------------------------------------------
Distance: 0.7203402519226074
{
  "answer": "Herring",
  "question": "The Bismarck type of this fish is made of fillets cured in vinegar, salt & onions",
  "round": "Double Jeopardy!"
}
-----------

In [21]:
client.close()